# DBT

The Data Build Tool (DBT) is a popular framework for building data manipulation systems.


## CLI

DBT is implemented in two versions:

- [`dbt-core`](https://pypi.org/project/dbt-core/): the python package that implements the CLI with which you can manipulate your project. Use `dbt` command to acces this tool.
- [DBT fusion](https://docs.getdbt.com/docs/fusion/about-fusion) new implementation of dbt that provides faster compilation of the project. Use `dbtf` command to access the fusion.

They have really similar interface which is discussed in this section.

For more check
- [Reference to the `dbt` CLI](https://docs.getdbt.com/reference/dbt-commands).
- [CLI](dbt/cli.ipynb) page.

## Structure

The DBT project suposes some kind of structure. There are corresponding directories/files:

- `/models`: the sql or python files that define what have to be done in the dbt project.
- `/seeds`: CSV-files, containing some additional information.
- `/tests`: unit-tests of the dbt project.
- `/maros`: reusable jinja templates.
- `/analysis`: files for Adhoc queries.
- `dbt_project.yml`: the main files for prject configuration.
- `profies.yml`: file that describes the credentials to connect to the database.

### Data layers

The dbt recommends organizing your transformation into three layers:

- **Staging**: the raw data received from datasources.
- **Intermediate**: calculations layer.
- **Marts**: for data marts (рус. Витрины данных).

Check more on project sctucture and naming conventions in the [How we structure out dbt prjects](https://docs.getdbt.com/best-practices/how-we-structure/1-guide-overview?version=1.12) guide from the dbt team.

## Models

In DBT, models are SQL scripts containing Jinja templates. DBT compiles these scripts into corresponding SQL code and then executes them. The results of the queries are wrapped in the corresponding DDL/DML and the results are put into the target database.

---

The following cell initialise the project.

In [1]:
#init
dbtf init --project-name knowledge --profile knowledge -q
cd knowledge

Info Created .vscode/extensions.json with dbt extension recommendation
Success Project created successfully!
Info Project name: knowledge
Info Project directory: knowledge
Validating profile inputs, adapters, and connection



The definition of the model:

In [10]:
# file knowledge/models/script.sql
SELECT 'some data' AS col, 'new data' AS col2

This "model" simply selects hard-coded values from the database.

In [11]:
dbtf run -q --select script

   Running [                    ] 0/1 1 in-progress                             
   Running [━━━━━━━━━━━━━━━━━━━━] 1/1             
   model:script [0.0s]                                                          


The table with same name then appears in the target PostgreSQL database.

In [12]:
PGPASSWORD=postgres psql -h localhost -U postgres -c "select * from public.script"

    col    |   col2   
-----------+----------
 some data | new data
(1 row)



## Reference

You can invoke one dbt object from another. This can be achieved using the `ref` function. Where the SQL syntax requires the data source, you can use the Jinja template `{{ ref('name.sql') }}`. The dbt will generate the corresponding dialect specific sql code.

---

The following cells define the project and the `model_a.sql` file we will reference as an example. 

In [11]:
#init
dbt init knowledge --profile knowledge -q
cd knowledge

In [31]:
#file knowledge/models/model_a.sql
SELECT * FROM (
VALUES
    ('hello', 1),
    ('world', 2)
) AS tab(col1, col2)

The following cell generates the `model_b.sql` that uses `ref('model_a')` as the datasource.

In [33]:
#file knowledge/models/model_b.sql
SELECT * FROM {{ ref('model_a') }} WHERE col2=2

The data generated by `model_b` corresponds to a subset of the `model_a`.

In [35]:
dbt show -q --select model_b.sql

| col1  | col2 |
| ----- | ---- |
| world |    2 |

